# v3 — simulator reconstruction

Same analysis as `exp-v3_key-results` but uses the **forward simulator** instead of the surrogate decoder for waveform reconstruction.

**Sections:** parameter scatter vs GT · simulator reconstruction (mmHg) · SV ablation (v3 vs v3_nosv, direct vs 1NN).

Inference path: cath lab input → encoder → flow → posterior mean (24 params) + HR routed from input → `Cv8EedSimulator.run_batch` → Prv/Pra/Pvp/Pap in mmHg.

In [ ]:
import json, sys, h5py
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

try:
    ROOT = Path(globals()['_dh'][0]).parent
    assert (ROOT / 'dataset.py').exists()
except:
    ROOT = Path('/home/sa4604/cv-dann-sbi')
sys.path.insert(0, str(ROOT))

SIM_DIR = Path('/home/sa4604/cv-sbi-spin/simulator-cv8Eed')
sys.path.insert(0, str(SIM_DIR))

from dataset import (
    load_stats, load_manifest, ReducedCVDataset,
    PARAM_KEYS_INFER, N_CHANNELS, T, WAVE_KEYS_CONT,
)
from models import LipschitzReducedAutoencoderEncoder
from cv8eed_sim import Cv8EedSimulator

WAVE_KEYS_REAL = ['Prv', 'Pra', 'Pvp', 'Pap']

device = torch.device('cuda:0')
print('device:', device)

In [ ]:
RUN        = 'exp-v3_encoder-lipschitz_dann_flow-maf5'
VERSION    = '3'
RUN_DIR    = ROOT / f'outputs/{RUN}'

REAL_DATA  = Path('/home/sa4604/real_data/onebeat_300patients')
SIM_ROOT   = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
STATS_PATH = ROOT / 'norm_stats.json'
SIM_EXE    = SIM_DIR / 'build_sim/bin/Cv8SimApp'

N_POSTERIOR_SAMPLES = 1000
N_SIM_PCA           = 500

stats    = load_stats(STATS_PATH)
manifest = load_manifest(SIM_ROOT / 'manifest_train.json')
lo_t = torch.tensor([manifest['config']['pvar_low'][k]  for k in PARAM_KEYS_INFER], dtype=torch.float32)
hi_t = torch.tensor([manifest['config']['pvar_high'][k] for k in PARAM_KEYS_INFER], dtype=torch.float32)
print('Prior bounds loaded for', len(PARAM_KEYS_INFER), 'parameters')

## Load models

In [ ]:
encoder = LipschitzReducedAutoencoderEncoder(latent_dim=128).to(device)
encoder.load_state_dict(torch.load(RUN_DIR / 'encoder.pt', map_location=device))
encoder.eval()
print('Encoder loaded:', encoder.describe())

flow_net = torch.load(RUN_DIR / 'flow_net.pt', map_location=device, weights_only=False)
flow_net.eval()
print('Flow loaded:', type(flow_net).__name__)

## Load real patients

In [ ]:
w = stats['waves']
p = stats['parameters']
wave_mean = torch.tensor([w[k]['mean'] for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
wave_std  = torch.tensor([w[k]['std']  for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
pas_mean, pas_std = w['Pas']['mean'], w['Pas']['std'] + 1e-8
vlv_std           = w['Vlv']['std'] + 1e-8
hr_mean,  hr_std  = p['HR']['mean'],  p['HR']['std']  + 1e-8

patients = []
for fpath in sorted(REAL_DATA.glob('*.h5')):
    beats_x, gt = [], {}
    with h5py.File(fpath, 'r') as f:
        for bk in sorted(f.keys()):
            if not bk.startswith('beat_'): continue
            g = f[bk]
            waves = np.stack([g[f'waves/{k}'][:].astype(np.float32) for k in WAVE_KEYS_REAL])
            wt    = (torch.from_numpy(waves) - wave_mean) / (wave_std + 1e-8)
            sbp   = float(g['summaries/sbp'][()])
            dbp   = float(g['summaries/dbp'][()])
            map_  = float(g['summaries/map'][()])
            sv    = float(g['summaries/sv'][()])
            hr    = float(g['parameters/HR'][()])
            sc = torch.tensor([
                (map_ - pas_mean) / pas_std,
                (sbp  - pas_mean) / pas_std,
                (dbp  - pas_mean) / pas_std,
                sv    / vlv_std,
                (hr   - hr_mean)  / hr_std,
            ], dtype=torch.float32)
            beats_x.append(torch.cat([wt.reshape(-1), sc]))
            if not gt:
                gt = {k: float(g[f'parameters/{k}'][()]) for k in f[bk]['parameters'].keys()}
                for sk in g['summaries'].keys():
                    gt[sk] = float(g[f'summaries/{sk}'][()])
                cov = g['covariates']
                gt['cohort'] = cov['cohort'][()].decode() if isinstance(cov['cohort'][()], bytes) else str(cov['cohort'][()])
                gt['id']     = cov['id'][()].decode()     if isinstance(cov['id'][()],     bytes) else str(cov['id'][()])
    if beats_x:
        x_beats = torch.stack(beats_x)
        patients.append(dict(file=fpath.stem, x_beats=x_beats, x_avg=x_beats.mean(0), gt=gt))

cohort_counts = {}
for pat in patients:
    c = pat['gt'].get('cohort', 'unknown')
    cohort_counts[c] = cohort_counts.get(c, 0) + 1
print(f'Loaded {len(patients)} patients, {sum(len(p["x_beats"]) for p in patients)} total beats')
print(f'Cohorts: {cohort_counts}')

In [ ]:
import time

cas_idx = PARAM_KEYS_INFER.index('Cas')
eap_idx = PARAM_KEYS_INFER.index('Eap')
rap_idx = PARAM_KEYS_INFER.index('Rap')
ras_idx = PARAM_KEYS_INFER.index('Ras')

print(f'Direct inference on {len(patients)} real patients (N={N_POSTERIOR_SAMPLES} samples each)...')

cas_d, eap_d, rap_d, ras_d, accept_rates, theta_means = [], [], [], [], [], []
rap_samples_all, ras_samples_all = [], []
t0 = time.time()

for pi, pat in enumerate(patients):
    x = pat['x_avg'].unsqueeze(0).to(device)
    with torch.no_grad():
        z       = encoder(x)
        samples = flow_net.sample((N_POSTERIOR_SAMPLES,), condition=z).squeeze(1).cpu()
    in_prior = ((samples >= lo_t) & (samples <= hi_t)).all(dim=1)
    accept   = in_prior.float().mean().item()
    accept_rates.append(accept)
    s_ok  = samples[in_prior] if in_prior.any() else samples
    means = s_ok.mean(0).numpy()
    theta_means.append(means)
    cas_d.append(means[cas_idx]); eap_d.append(means[eap_idx])
    rap_d.append(means[rap_idx]); ras_d.append(means[ras_idx])
    rap_samples_all.append(samples[:, rap_idx].numpy())
    ras_samples_all.append(samples[:, ras_idx].numpy())
    if pi % 100 == 0 or pi < 3:
        print(f'  [{pi:3d}] accept={accept:.3f}  '
              f'Cas={cas_d[-1]:.3f} (GT={pat["gt"].get("Cas", float("nan")):.3f})  '
              f'({time.time()-t0:.0f}s)')

cas_d = np.array(cas_d); eap_d = np.array(eap_d)
rap_d = np.array(rap_d); ras_d = np.array(ras_d)
accept_rates = np.array(accept_rates)
theta_means  = np.stack(theta_means)   # (N_patients, 24) in physical units
hr_all       = np.array([float(pat['gt'].get('HR', np.nan)) for pat in patients])
rap_samples_all = np.stack(rap_samples_all)
ras_samples_all = np.stack(ras_samples_all)
print(f'\nMean acceptance: {accept_rates.mean():.3f}  '
      f'({(accept_rates > 0).sum()}/{len(patients)} patients with nonzero acceptance)')

## Parameter scatter — posterior mean vs GT

In [ ]:
from scipy.stats import pearsonr

cas_gt = np.array([pat['gt'].get('Cas', np.nan) for pat in patients])
eap_gt = np.array([pat['gt'].get('Eap', np.nan) for pat in patients])
rap_gt = np.array([pat['gt'].get('PVR', np.nan) for pat in patients])
ras_gt = np.array([pat['gt'].get('SVR', np.nan) for pat in patients])

fig, axes = plt.subplots(1, 5, figsize=(28, 5))
for ax, pred, gt, name, color in [
    (axes[0], cas_d, cas_gt, 'Cas',       'steelblue'),
    (axes[1], eap_d, eap_gt, 'Eap',       'tomato'),
    (axes[2], rap_d, rap_gt, 'Rap (PVR)', 'mediumseagreen'),
    (axes[3], ras_d, ras_gt, 'Ras (SVR)', 'orchid'),
]:
    valid = ~np.isnan(pred) & ~np.isnan(gt) & (gt >= 0)
    if valid.any():
        lo = np.nanmin([gt[valid].min(), pred[valid].min()])
        hi = np.nanmax([gt[valid].max(), pred[valid].max()])
        ax.scatter(gt[valid], pred[valid], s=15, alpha=0.7, color=color, marker='^')
        ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
        mape = np.nanmean(np.abs(pred[valid] - gt[valid]) / (np.abs(gt[valid]) + 1e-9)) * 100
        r2   = pearsonr(gt[valid], pred[valid])[0] ** 2
        ax.set_title(f'{name}  MAPE={mape:.1f}%  R²={r2:.3f}\n({valid.sum()}/{len(patients)} pts)')
    else:
        ax.set_title(f'{name}  0 valid')
    ax.set_xlabel(f'{name} measured'); ax.set_ylabel(f'{name} posterior mean')

ax = axes[4]
ax.bar(range(len(patients)), accept_rates, color='orchid', alpha=0.8)
ax.axhline(accept_rates.mean(), color='black', linestyle='--', linewidth=1,
           label=f'mean {accept_rates.mean():.3f}')
ax.set_xlabel('Patient'); ax.set_ylabel('Acceptance rate')
ax.set_title('Per-patient acceptance rate'); ax.legend(fontsize=8)

fig.suptitle(f'Direct inference — {RUN}', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\n{"Parameter":<12} {"MAPE":>8} {"R²":>7}')
print('-' * 30)
for pred, gt, name in [
    (cas_d, cas_gt, 'Cas'), (eap_d, eap_gt, 'Eap'),
    (rap_d, rap_gt, 'Rap(PVR)'), (ras_d, ras_gt, 'Ras(SVR)'),
]:
    valid = ~np.isnan(pred) & ~np.isnan(gt) & (gt >= 0)
    if valid.sum() > 1:
        mape = np.nanmean(np.abs(pred[valid] - gt[valid]) / (np.abs(gt[valid]) + 1e-9)) * 100
        r2   = pearsonr(gt[valid], pred[valid])[0] ** 2
        print(f'{name:<12} {mape:>7.1f}%  {r2:>6.3f}')

## Simulator reconstruction — physical units (mmHg)

Posterior mean theta (24 params, physical units) + HR routed from the cath lab input → `Cv8EedSimulator.run_batch` → Prv/Pra/Pvp/Pap vs real observed waveforms.

- **Plot 1** — posterior mean: 8 example patients × 4 channels
- **Plot 2** — P5–P95 envelope from posterior samples, 8 patients
- **Table** — per-channel MSE/RMSE (mmHg) and Pearson r over all 802 patients

In [ ]:
sim = Cv8EedSimulator(executable=str(SIM_EXE))
print(f'Simulator backend: {sim.backend}')

N_T        = 201
N_EXAMPLES = 8
t_axis     = np.arange(N_T)

wave_mean_arr = np.array([stats['waves'][k]['mean'] for k in WAVE_KEYS_REAL], dtype=np.float32)
wave_std_arr  = np.array([stats['waves'][k]['std']  for k in WAVE_KEYS_REAL], dtype=np.float32)

def to_mmhg(z):
    """(N, 4, T) z-scored -> mmHg."""
    return z * wave_std_arr[None, :, None] + wave_mean_arr[None, :, None]

SIM_NB_DIR = ROOT / 'outputs/exp-v3_key-results_simulator'
SIM_NB_DIR.mkdir(parents=True, exist_ok=True)
RECON_CACHE = SIM_NB_DIR / 'recon_waves.npz'

if RECON_CACHE.exists():
    print(f'Loading cached reconstruction from {RECON_CACHE}')
    _d = np.load(RECON_CACHE, allow_pickle=True)
    waves_cat   = _d['waves_cat']
    success_cat = _d['success_cat'].astype(bool)
    var_names   = list(_d['var_names'])
    print(f'  waves_cat: {waves_cat.shape}  success: {success_cat.sum()}/{len(success_cat)}')
else:
    lo_np = lo_t.numpy(); hi_np = hi_t.numpy()

    theta_means_raw = []
    hr_means_raw    = []
    t0 = time.time()
    print(f'Computing posterior means for {len(patients)} patients...')
    for pi, pat in enumerate(patients):
        x = pat['x_avg'].unsqueeze(0).to(device)
        with torch.no_grad():
            z = encoder(x)
            s_raw = flow_net.sample((N_POSTERIOR_SAMPLES,), condition=z).squeeze(1).cpu().numpy()
        # flow_net.sample() returns physical-space params — clip to prior, no de-normalization
        m_phy = np.clip(s_raw.mean(axis=0), lo_np, hi_np)
        theta_means_raw.append(m_phy)
        hr_means_raw.append(pat['x_avg'][-1].item() * hr_std + hr_mean)
        if pi % 100 == 0:
            print(f'  {pi}/{len(patients)}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    print(f'\n  inference done ({time.time()-t0:.0f}s)')
    theta_means_raw = np.stack(theta_means_raw)

    param_list = []
    for pi in range(len(patients)):
        d = {k: float(theta_means_raw[pi, j]) for j, k in enumerate(PARAM_KEYS_INFER)}
        d['HR'] = float(hr_means_raw[pi])
        param_list.append(d)

    from tqdm.auto import tqdm as _tqdm
    CHUNK_SIZE = 1
    chunks = [param_list[i:i+CHUNK_SIZE] for i in range(0, len(param_list), CHUNK_SIZE)]
    all_waves, all_success = [], []
    var_names = None
    n_vars_known = None
    n_done = 0
    t_start = time.time()

    with _tqdm(total=len(param_list), unit='sim') as pbar:
        for chunk in chunks:
            try:
                b = sim.run_batch(chunk, threads=8, timeout=30)
                if var_names is None:
                    var_names    = b['var_names']
                    n_vars_known = len(var_names)
                nv = n_vars_known or 1
                w  = b['waves'] if b['waves'] is not None else np.full((len(chunk), N_T, nv), np.nan)
                all_waves.append(w)
                all_success.append(b['status']['success'].astype(bool))
            except TimeoutError:
                nv = n_vars_known or 1
                all_waves.append(np.full((len(chunk), N_T, nv), np.nan))
                all_success.append(np.zeros(len(chunk), dtype=bool))
                pbar.write(f'  timeout: skipped {len(chunk)} sims')
            n_done += len(chunk)
            elapsed = time.time() - t_start
            rate    = n_done / elapsed
            eta     = (len(param_list) - n_done) / rate if rate > 0 else 0
            pbar.set_postfix({'done': n_done, 'rate': f'{rate:.1f}/s', 'ETA': f'{eta:.0f}s'})
            pbar.update(len(chunk))

    waves_cat   = np.concatenate(all_waves,   axis=0)
    success_cat = np.concatenate(all_success, axis=0)
    np.savez(RECON_CACHE, waves_cat=waves_cat, success_cat=success_cat,
             var_names=np.array(var_names))
    print(f'Saved to {RECON_CACHE}  (success: {success_cat.sum()}/{len(success_cat)})')

ch_idxs    = [var_names.index(k) for k in WAVE_KEYS_REAL]
recon_phys = waves_cat[:, :, ch_idxs].transpose(0, 2, 1)  # (802, 4, 201) mmHg
failed_mask = ~success_cat
recon_phys[failed_mask] = np.nan

In [ ]:
real_waves_all = np.stack([
    pat['x_avg'][:len(WAVE_KEYS_REAL) * N_T].numpy().reshape(len(WAVE_KEYS_REAL), N_T)
    for pat in patients
])
real_phys = to_mmhg(real_waves_all)  # (802, 4, 201) mmHg

ex_idxs = [i for i in range(len(patients)) if success_cat[i]][:N_EXAMPLES]

fig, axes = plt.subplots(len(WAVE_KEYS_REAL), N_EXAMPLES,
                         figsize=(4 * N_EXAMPLES, 3 * len(WAVE_KEYS_REAL)), sharey='row')
for ci, ch in enumerate(WAVE_KEYS_REAL):
    for j, pi in enumerate(ex_idxs):
        ax = axes[ci, j]
        ax.plot(t_axis, recon_phys[pi, ci], color='tomato',    linewidth=1.5, label='sim recon')
        ax.plot(t_axis, real_phys[pi, ci],  color='steelblue', linewidth=1.5, label='real')
        if ci == 0: ax.set_title(f'Pat {pi}', fontsize=7)
        if j == 0:  ax.set_ylabel(f'{ch} (mmHg)', fontsize=7)
        if ci == len(WAVE_KEYS_REAL) - 1: ax.set_xlabel('time step', fontsize=7)
        ax.tick_params(labelsize=6)
axes[0, 0].legend(fontsize=6)
fig.suptitle(f'Posterior mean → simulator vs real — {RUN}', fontsize=11)
plt.tight_layout(); plt.show()

# Per-channel RMSE table
from scipy.stats import pearsonr
ok = success_cat
print(f'\n{"Channel":<8} {"RMSE (mmHg)":>14} {"Pearson r":>12} {"n":>5}')
print('-' * 43)
for ci, ch in enumerate(WAVE_KEYS_REAL):
    r_ch = recon_phys[ok, ci, :]
    t_ch = real_phys[ok, ci, :]
    rmse = np.sqrt(np.mean((r_ch - t_ch) ** 2))
    rs   = [pearsonr(r_ch[i], t_ch[i])[0] for i in range(len(r_ch))]
    print(f'{ch:<8} {rmse:>14.2f} {np.mean(rs):>12.3f} {ok.sum():>5}')

In [ ]:
import time

N_SIM_EVAL = 1000

manifest_test = load_manifest(SIM_ROOT / 'manifest_test.json')
ds_test = ReducedCVDataset(str(SIM_ROOT / 'test'), manifest_test['index'][:N_SIM_EVAL], stats)
theta_test, x_test = zip(*[ds_test[i] for i in range(len(ds_test))])
theta_test = torch.stack(list(theta_test))  # (N, 24)
x_test     = torch.stack(list(x_test))      # (N, 809)
print(f'Test sims: {tuple(theta_test.shape)}  obs: {tuple(x_test.shape)}')

---
## SV ablation: v3_nosv — simulator-based

Direct inference with nosv encoder → posterior mean (24 params) + HR → simulator → Vrv/Vlv SV (max − min in mL) vs thermodilution GT.

In [ ]:
OUT_V3_NOSV = ROOT / 'outputs/exp-v3_nosv_encoder-lipschitz_dann_flow-maf5'

enc_nosv = LipschitzReducedAutoencoderEncoder(latent_dim=128, n_scalars=4).to(device)
enc_nosv.load_state_dict(torch.load(OUT_V3_NOSV / 'encoder.pt', map_location=device))
enc_nosv.eval()
flow_nosv = torch.load(OUT_V3_NOSV / 'flow_net.pt', map_location=device, weights_only=False)
flow_nosv.eval()
print('v3_nosv encoder:', enc_nosv.describe())

# 808-dim obs: drop SV scalar at position 807, keep HR at 808
for pat in patients:
    x809 = pat['x_beats']
    pat['x_beats_nosv'] = torch.cat([x809[:, :807], x809[:, 808:]], dim=1)
    pat['x_avg_nosv']   = pat['x_beats_nosv'].mean(0)
print(f'nosv obs dim: {patients[0]["x_avg_nosv"].shape[0]}')

In [ ]:
from tqdm.auto import tqdm as _tqdm
from scipy.stats import pearsonr

sv_gt  = np.array([pat['gt']['sv'] for pat in patients])
hr_arr = np.array([pat['gt']['HR'] for pat in patients])

def metrics_sv(sv_pred, sv_true):
    mask = np.isfinite(sv_pred) & np.isfinite(sv_true)
    if mask.sum() < 2:
        return float('nan'), float('nan'), mask.sum()
    r2   = pearsonr(sv_true[mask], sv_pred[mask])[0] ** 2
    mape = np.mean(np.abs(sv_pred[mask] - sv_true[mask]) / (np.abs(sv_true[mask]) + 1e-9)) * 100
    return r2, mape, int(mask.sum())

NOSV_RECON_CACHE = SIM_NB_DIR / 'recon_waves_nosv.npz'

if NOSV_RECON_CACHE.exists():
    print(f'Loading cached nosv reconstruction from {NOSV_RECON_CACHE}')
    _d = np.load(NOSV_RECON_CACHE, allow_pickle=True)
    waves_cat_nosv   = _d['waves_cat']
    success_cat_nosv = _d['success_cat'].astype(bool)
    var_names_nosv   = list(_d['var_names'])
    print(f'  waves_cat: {waves_cat_nosv.shape}  success: {success_cat_nosv.sum()}/{len(success_cat_nosv)}')
else:
    lo_np = lo_t.numpy(); hi_np = hi_t.numpy()

    theta_means_nosv = []
    hr_means_nosv    = []
    t0 = time.time()
    print(f'Computing nosv posterior means for {len(patients)} patients...')
    for pi, pat in enumerate(patients):
        x = pat['x_avg_nosv'].unsqueeze(0).to(device)
        with torch.no_grad():
            z = enc_nosv(x)
            s_raw = flow_nosv.sample((N_POSTERIOR_SAMPLES,), condition=z).squeeze(1).cpu().numpy()
        m_phy = np.clip(s_raw.mean(axis=0), lo_np, hi_np)
        theta_means_nosv.append(m_phy)
        hr_means_nosv.append(pat['x_avg_nosv'][-1].item() * hr_std + hr_mean)
        if pi % 100 == 0:
            print(f'  {pi}/{len(patients)}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    print(f'\n  inference done ({time.time()-t0:.0f}s)')
    theta_means_nosv = np.stack(theta_means_nosv)

    param_list_nosv = []
    for pi in range(len(patients)):
        d = {k: float(theta_means_nosv[pi, j]) for j, k in enumerate(PARAM_KEYS_INFER)}
        d['HR'] = float(hr_means_nosv[pi])
        param_list_nosv.append(d)

    all_waves, all_success = [], []
    var_names_nosv = None
    n_vars_known   = None
    n_done = 0
    t_start = time.time()

    with _tqdm(total=len(param_list_nosv), unit='sim') as pbar:
        for chunk in [param_list_nosv[i:i+1] for i in range(len(param_list_nosv))]:
            try:
                b = sim.run_batch(chunk, threads=8, timeout=30)
                if var_names_nosv is None:
                    var_names_nosv = b['var_names']
                    n_vars_known   = len(var_names_nosv)
                nv = n_vars_known or 1
                w  = b['waves'] if b['waves'] is not None else np.full((1, N_T, nv), np.nan)
                all_waves.append(w)
                all_success.append(b['status']['success'].astype(bool))
            except TimeoutError:
                nv = n_vars_known or 1
                all_waves.append(np.full((1, N_T, nv), np.nan))
                all_success.append(np.zeros(1, dtype=bool))
            n_done += 1
            elapsed = time.time() - t_start
            rate    = n_done / elapsed
            eta     = (len(param_list_nosv) - n_done) / rate if rate > 0 else 0
            pbar.set_postfix({'done': n_done, 'rate': f'{rate:.1f}/s', 'ETA': f'{eta:.0f}s'})
            pbar.update(1)

    waves_cat_nosv   = np.concatenate(all_waves,   axis=0)
    success_cat_nosv = np.concatenate(all_success, axis=0)
    np.savez(NOSV_RECON_CACHE, waves_cat=waves_cat_nosv, success_cat=success_cat_nosv,
             var_names=np.array(var_names_nosv))
    print(f'Saved to {NOSV_RECON_CACHE}  (success: {success_cat_nosv.sum()}/{len(success_cat_nosv)})')

In [ ]:
ok_nosv = success_cat_nosv
vrv_idx = var_names_nosv.index('Vrv')
vlv_idx = var_names_nosv.index('Vlv')

sv_nosv_vrv = np.where(ok_nosv,
    waves_cat_nosv[:, :, vrv_idx].max(axis=1) - waves_cat_nosv[:, :, vrv_idx].min(axis=1),
    np.nan)
sv_nosv_vlv = np.where(ok_nosv,
    waves_cat_nosv[:, :, vlv_idx].max(axis=1) - waves_cat_nosv[:, :, vlv_idx].min(axis=1),
    np.nan)

print(f'nosv: Vrv valid={np.isfinite(sv_nosv_vrv).sum()}, Vlv valid={np.isfinite(sv_nosv_vlv).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, label, sv_pred, color in [
    (axes[0], 'nosv — Vrv', sv_nosv_vrv, 'steelblue'),
    (axes[1], 'nosv — Vlv', sv_nosv_vlv, 'tomato'),
]:
    r2, mape, n = metrics_sv(sv_pred, sv_gt)
    mask = np.isfinite(sv_pred)
    if mask.sum() > 1:
        lo = min(sv_gt[mask].min(), sv_pred[mask].min())
        hi = max(sv_gt[mask].max(), sv_pred[mask].max())
        ax.scatter(sv_gt[mask], sv_pred[mask], s=15, alpha=0.65, color=color)
        ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    ax.set_xlabel('GT SV — thermodilution (mL)')
    ax.set_ylabel('Simulator SV — posterior mean (mL)')
    ax.set_title(f'{label}\nR²={r2:.3f}  MAPE={mape:.1f}%  n={n}', fontsize=10)
fig.suptitle('SV ablation — v3_nosv direct inference — simulator', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\n{"":20} {"R²":>6} {"MAPE":>8} {"n":>5}')
print('-' * 43)
for label, sv_pred in [('nosv — Vrv', sv_nosv_vrv), ('nosv — Vlv', sv_nosv_vlv)]:
    r2, mape, n = metrics_sv(sv_pred, sv_gt)
    print(f'{label:<20} {r2:>6.3f} {mape:>7.1f}%  {n:>4}')

In [ ]:
SAMP_1NN_NOSV_CACHE = SIM_NB_DIR / 'samp_1nn_nosv.npz'

if SAMP_1NN_NOSV_CACHE.exists():
    print('Loading 1NN nosv samples from cache...')
    samp_1nn_nosv = np.load(SAMP_1NN_NOSV_CACHE)['samp_1nn_nosv']
else:
    x_test_nosv = torch.cat([x_test[:, :807], x_test[:, 808:]], dim=1)
    print('Encoding test sims (nosv)...')
    with torch.no_grad():
        z_sims_nosv = torch.cat([enc_nosv(x_test_nosv[i:i+256].to(device))
                                  for i in range(0, len(x_test_nosv), 256)], dim=0).cpu()

    samps = []
    t0 = time.time()
    for pi, pat in enumerate(patients):
        with torch.no_grad():
            z_real = enc_nosv(pat['x_avg_nosv'].unsqueeze(0).to(device)).cpu()
            i_nn   = ((z_sims_nosv - z_real) ** 2).sum(dim=1).argmin().item()
            z_nn   = z_sims_nosv[i_nn:i_nn+1].to(device)
            s = flow_nosv.sample((N_POSTERIOR_SAMPLES,), condition=z_nn).squeeze(1).cpu()
        samps.append(s.numpy())
        if pi % 100 == 0:
            print(f'  {pi}/{len(patients)}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    print(f'  done ({time.time()-t0:.0f}s)')
    samp_1nn_nosv = np.stack(samps)
    np.savez(SAMP_1NN_NOSV_CACHE, samp_1nn_nosv=samp_1nn_nosv)
    print(f'Saved to {SAMP_1NN_NOSV_CACHE}')

print(f'samp_1nn_nosv: {samp_1nn_nosv.shape}')

In [ ]:
NOSV_1NN_RECON_CACHE = SIM_NB_DIR / 'recon_waves_1nn_nosv.npz'

if NOSV_1NN_RECON_CACHE.exists():
    print(f'Loading cached 1NN nosv reconstruction from {NOSV_1NN_RECON_CACHE}')
    _d = np.load(NOSV_1NN_RECON_CACHE, allow_pickle=True)
    waves_cat_1nn_nosv   = _d['waves_cat']
    success_cat_1nn_nosv = _d['success_cat'].astype(bool)
    var_names_1nn_nosv   = list(_d['var_names'])
    print(f'  waves_cat: {waves_cat_1nn_nosv.shape}  success: {success_cat_1nn_nosv.sum()}/{len(success_cat_1nn_nosv)}')
else:
    lo_np = lo_t.numpy(); hi_np = hi_t.numpy()

    theta_means_1nn = np.stack([
        np.clip(samp_1nn_nosv[pi].mean(axis=0), lo_np, hi_np)
        for pi in range(len(patients))
    ])
    hr_means_1nn = [pat['x_avg_nosv'][-1].item() * hr_std + hr_mean for pat in patients]

    param_list_1nn = []
    for pi in range(len(patients)):
        d = {k: float(theta_means_1nn[pi, j]) for j, k in enumerate(PARAM_KEYS_INFER)}
        d['HR'] = float(hr_means_1nn[pi])
        param_list_1nn.append(d)

    all_waves, all_success = [], []
    var_names_1nn_nosv = None
    n_vars_known = None
    n_done = 0
    t_start = time.time()

    with _tqdm(total=len(param_list_1nn), unit='sim') as pbar:
        for chunk in [param_list_1nn[i:i+1] for i in range(len(param_list_1nn))]:
            try:
                b = sim.run_batch(chunk, threads=8, timeout=30)
                if var_names_1nn_nosv is None:
                    var_names_1nn_nosv = b['var_names']
                    n_vars_known = len(var_names_1nn_nosv)
                nv = n_vars_known or 1
                w  = b['waves'] if b['waves'] is not None else np.full((1, N_T, nv), np.nan)
                all_waves.append(w)
                all_success.append(b['status']['success'].astype(bool))
            except TimeoutError:
                nv = n_vars_known or 1
                all_waves.append(np.full((1, N_T, nv), np.nan))
                all_success.append(np.zeros(1, dtype=bool))
            n_done += 1
            elapsed = time.time() - t_start
            rate = n_done / elapsed
            eta  = (len(param_list_1nn) - n_done) / rate if rate > 0 else 0
            pbar.set_postfix({'done': n_done, 'rate': f'{rate:.1f}/s', 'ETA': f'{eta:.0f}s'})
            pbar.update(1)

    waves_cat_1nn_nosv   = np.concatenate(all_waves,   axis=0)
    success_cat_1nn_nosv = np.concatenate(all_success, axis=0)
    np.savez(NOSV_1NN_RECON_CACHE, waves_cat=waves_cat_1nn_nosv, success_cat=success_cat_1nn_nosv,
             var_names=np.array(var_names_1nn_nosv))
    print(f'Saved to {NOSV_1NN_RECON_CACHE}  (success: {success_cat_1nn_nosv.sum()}/{len(success_cat_1nn_nosv)})')

In [ ]:
ok_1nn = success_cat_1nn_nosv
vrv_idx = var_names_1nn_nosv.index('Vrv')
vlv_idx = var_names_1nn_nosv.index('Vlv')

sv_1nn_nosv_vrv = np.where(ok_1nn,
    waves_cat_1nn_nosv[:, :, vrv_idx].max(axis=1) - waves_cat_1nn_nosv[:, :, vrv_idx].min(axis=1),
    np.nan)
sv_1nn_nosv_vlv = np.where(ok_1nn,
    waves_cat_1nn_nosv[:, :, vlv_idx].max(axis=1) - waves_cat_1nn_nosv[:, :, vlv_idx].min(axis=1),
    np.nan)

print(f'1NN nosv: Vrv valid={np.isfinite(sv_1nn_nosv_vrv).sum()}, Vlv valid={np.isfinite(sv_1nn_nosv_vlv).sum()}')

cases = [
    (0, 0, 'nosv direct — Vrv', sv_nosv_vrv,     'steelblue'),
    (0, 1, 'nosv 1NN — Vrv',    sv_1nn_nosv_vrv, 'seagreen'),
    (1, 0, 'nosv direct — Vlv', sv_nosv_vlv,     'tomato'),
    (1, 1, 'nosv 1NN — Vlv',    sv_1nn_nosv_vlv, 'mediumpurple'),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey='row')
for row, col, label, sv_pred, color in cases:
    ax = axes[row, col]
    r2, mape, n = metrics_sv(sv_pred, sv_gt)
    mask = np.isfinite(sv_pred)
    if mask.sum() > 1:
        lo = min(sv_gt[mask].min(), sv_pred[mask].min())
        hi = max(sv_gt[mask].max(), sv_pred[mask].max())
        ax.scatter(sv_gt[mask], sv_pred[mask], s=15, alpha=0.65, color=color)
        ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    ax.set_xlabel('GT SV — thermodilution (mL)')
    ax.set_ylabel('Simulator SV — posterior mean (mL)')
    ax.set_title(f'{label}\nR²={r2:.3f}  MAPE={mape:.1f}%  n={n}', fontsize=10)
fig.suptitle('SV ablation — v3_nosv: direct vs 1NN — simulator', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\n{"":24} {"R²":>6} {"MAPE":>8} {"n":>5}')
print('-' * 47)
for _, _, label, sv_pred, _ in cases:
    r2, mape, n = metrics_sv(sv_pred, sv_gt)
    print(f'{label:<24} {r2:>6.3f} {mape:>7.1f}%  {n:>4}')